# SDK: Chat, prompts y output estructurado

## 0. Reutilizar credenciales
> Copiar las variables del notebook anterior o volver a definirlas.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

WX_API_KEY        = os.getenv("WX_API_KEY", "")
WX_PROJECT_ID     = os.getenv("WX_PROJECT_ID", "")
WX_URL            = os.getenv("WX_URL", "https://us-south.ml.cloud.ibm.com")

print("Credenciales cargadas")

Credenciales cargadas


In [2]:
from ibm_watsonx_ai import APIClient, Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as Params

credentials = Credentials(url=WX_URL, api_key=WX_API_KEY)
client = APIClient(credentials)

model = ModelInference(
    model_id='meta-llama/llama-3-3-70b-instruct',
    api_client=client,
    project_id=WX_PROJECT_ID,
    params={Params.TEMPERATURE: 0.1}
)

print('Cliente listo')

Cliente listo


## 1. Formato chat — system + user + historial

In [3]:
messages = [
    {
        'role': 'system',
        'content': 'Sos un asistente técnico experto en watsonx. '
                   'Respondés de forma concisa y siempre en español.'
    },
    {
        'role': 'user',
        'content': '¿Cuál es la diferencia entre un modelo instruct y un modelo chat?'
    }
]

response = model.chat(messages=messages)
print('Respuesta:')
print(response['choices'][0]['message']['content'])

Respuesta:
Un modelo instruct es entrenado para seguir instrucciones y realizar tareas específicas, mientras que un modelo chat es entrenado para mantener conversaciones y responder a preguntas de manera más libre y natural.


## 2. Output estructurado en JSON
Patrón fundamental para agentes: que devuelvan datos procesables, no texto.

In [4]:
import json

messages_json = [
    {
        'role': 'system',
        'content': '''Clasificás tickets de soporte técnico.
Respondé SIEMPRE con un JSON válido con este esquema:
{"categoria": str, "prioridad": "baja|media|alta|critica", "resumen": str}
No incluyas texto fuera del JSON.'''
    },
    {
        'role': 'user',
        'content': 'La app de producción no levanta después del deploy de las 14hs.'
    }
]

response = model.chat(messages=messages_json)
raw = response['choices'][0]['message']['content']

print('Raw response:', raw)
print()

# Parsear y verificar
try:
    parsed = json.loads(raw)
    print('JSON válido:')
    print(json.dumps(parsed, indent=2, ensure_ascii=False))
    assert 'categoria' in parsed
    assert parsed['prioridad'] in ['baja', 'media', 'alta', 'critica']
    print('\nSchema correcto')
except json.JSONDecodeError as e:
    print('El modelo no devolvió JSON válido:', e)
    print(' Revisá el system prompt o bajá la temperatura a 0.0')

Raw response: {"categoria": "Incidente", "prioridad": "critica", "resumen": "Falló el deploy de la app de producción a las 14hs"}

JSON válido:
{
  "categoria": "Incidente",
  "prioridad": "critica",
  "resumen": "Falló el deploy de la app de producción a las 14hs"
}

Schema correcto


## 3. Ejercicio libre — modificar el system prompt
> Modificá el system prompt para clasificar en categorías distintas.  
> Probá agregar un campo extra al schema JSON (ej: `area_responsable`).

In [5]:
# Tu turno — modificá este template
mi_system = '''Clasificás ... # describí qué hace tu agente
Respondé con este schema:
{"campo1": str, "campo2": str}'''

mi_input = '...tu caso de prueba...'

# messages_custom = [
#     {'role': 'system', 'content': mi_system},
#     {'role': 'user', 'content': mi_input}
# ]
# response = model.chat(messages=messages_custom)
# print(response['choices'][0]['message']['content'])